In [2]:
#This code block makes a dictionary from the mzml file. These dictionaries can then be made for any file type. 
from pyopenms import MSExperiment, MzMLFile
import numpy as np
import plotly.express as px
import pandas as pd

def parse_mzml_to_dict(filename):
    # Load the mzML file
    exp = MSExperiment()
    MzMLFile().load(filename, exp)

    # Initialize the dictionary
    ms_dict = {"MS1": {}, "MS2": {}}

    # Iterate through spectra in the experiment
    for spectrum in exp:
        # Get the MS level
        ms_level = spectrum.getMSLevel()

        # Only process MS1 and MS2 spectra
        if ms_level in [1, 2]:
            # Get the retention time
            rt = spectrum.getRT()/60

            # Get m/z and intensity arrays
            mz_array, intensity_array = spectrum.get_peaks()

            # Store in the dictionary
            key = f"MS{ms_level}"
            ms_dict[key][rt] = {
                "m/z": np.array(mz_array),
                "intensity": np.array(intensity_array)
            }

    return ms_dict

# Usage
filename = "/Users/andrewbasalla/Code/MS_Viewer/MS_Viewer/data/022824_A_BSA_01.mzML"
result_dict = parse_mzml_to_dict(filename)


Determination of memory status is not supported on this 
 platform, measuring for memoryleaks will never fail


In [103]:
#Selecting retention time
value = 11.98
ms1_idx = (result_dict['MS1'][value] if value in result_dict['MS1'] else result_dict['MS1'][min(result_dict['MS1'].keys(), key=lambda x: abs(x-value))])
print(ms1_idx)

{'m/z': array([ 371.2708144 ,  371.271888  ,  371.27296161, ..., 1515.10751382,
       1515.11636444, 1515.12521513]), 'intensity': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32)}


In [ ]:
fig_ms1 = px.line(x=ms1_idx['m/z'], y=ms1_idx['intensity'])
fig_ms1.show()

In [ ]:
def generate_tic(ms_dict):
    """
    Generates a Total Ion Chromatogram (TIC) that includes both MS1 and MS2 data.
    
    Parameters:
    ms_dict (dict): The dictionary containing MS1 and MS2 data with retention time and intensity values.
    
    Returns:
    tuple: Two arrays, one for retention times and one for the summed intensities at each retention time.
    """
    # Initialize dictionaries to store summed intensities
    retention_times = sorted(set(list(ms_dict["MS1"].keys()) + list(ms_dict["MS2"].keys())))
    total_intensities = []

    # Sum intensities for MS1 and MS2 at each unique retention time
    for rt in retention_times:
        ms1_intensity = np.sum(ms_dict["MS1"].get(rt, {"intensity": []})["intensity"])
        ms2_intensity = np.sum(ms_dict["MS2"].get(rt, {"intensity": []})["intensity"])
        total_intensity = ms1_intensity + ms2_intensity
        total_intensities.append(total_intensity)

    return np.array(retention_times), np.array(total_intensities)

def plot_tic(retention_times, total_intensities):
    """
    Plots the Total Ion Chromatogram (TIC) using Plotly, combining MS1 and MS2 data.
    
    Parameters:
    retention_times (array): Array of retention times.
    total_intensities (array): Array of total ion intensities (combined MS1 and MS2).
    """
    # Create a DataFrame for plotting
    tic_data = {"Retention Time": retention_times, "Total Intensity": total_intensities}
    
    # Plot TIC
    fig = px.line(tic_data, x="Retention Time", y="Total Intensity")
    fig.update_xaxes(title="Retention Time (min)")
    fig.update_yaxes(title="Total Intensity")
    fig.show()


rt_combined, total_intensity_combined = generate_tic(result_dict)

plot_tic(rt_combined, total_intensity_combined)

In [ ]:
def generate_bpc(ms_dict, ms_level="MS1"):
    """
    Generates the Base Peak Chromatogram (BPC) for the specified MS level.
    
    Parameters:
    ms_dict (dict): The dictionary generated from the mzML file with retention time and intensity data.
    ms_level (str): The MS level to generate BPC for (default is "MS1").
    
    Returns:
    tuple: Two arrays, one for retention times and one for the base peak intensities at each retention time.
    """
    # Extract the MS1 or MS2 data
    ms_data = ms_dict.get(ms_level, {})

    # Prepare arrays for retention time and base peak intensity
    retention_times = []
    base_peak_intensities = []

    # Find the base peak (max intensity) for each retention time
    for rt, data in ms_data.items():
        # Check if there are intensities in the current data
        if len(data["intensity"]) > 0:
            base_peak_intensity = np.max(data["intensity"])
        else:
            base_peak_intensity = 0  # No intensity data for this retention time

        retention_times.append(rt)
        base_peak_intensities.append(base_peak_intensity)

    return np.array(retention_times), np.array(base_peak_intensities)

def plot_bpc(retention_times, base_peak_intensities):
    """
    Plots the Base Peak Chromatogram (BPC) using Plotly.
    
    Parameters:
    retention_times (array): Array of retention times.
    base_peak_intensities (array): Array of base peak intensities.
    """
    # Create a DataFrame for plotting
    bpc_data = {"Retention Time": retention_times, "Base Peak Intensity": base_peak_intensities}
    
    # Plot BPC
    fig = px.line(bpc_data, x="Retention Time", y="Base Peak Intensity")
    fig.update_xaxes(title="Retention Time (min)")
    fig.update_yaxes(title="Base Peak Intensity")
    fig.show()

# Usage
rt_bpc, base_intensity_bpc = generate_bpc(result_dict, ms_level="MS1")
plot_bpc(rt_bpc, base_intensity_bpc)

In [3]:
def generate_xic(result_dict, mz_range):
    """
    Generates an Extracted Ion Chromatogram (XIC) for a specified m/z range.
    
    Parameters:
    result_dict (dict): The dictionary with MS1 and MS2 data, containing retention time, m/z, and intensity values.
    mz_range (tuple): A tuple (min_mz, max_mz) defining the m/z range for the XIC.
    
    Returns:
    tuple: Two arrays, one for retention times and one for the summed intensities in the specified m/z range.
    """
    ms1_data = result_dict.get("MS1", {})

    # Prepare arrays for retention times and extracted intensities
    retention_times = []
    xic_intensities = []

    # Iterate over each retention time in MS1 data
    for rt, data in ms1_data.items():
        mz_values = data["m/z"]
        intensity_values = data["intensity"]

        # Find indices where m/z values are within the specified range
        mask = (mz_values >= mz_range[0]) & (mz_values <= mz_range[1])
        
        # Sum intensities within the m/z range for this retention time
        xic_intensity = np.sum(intensity_values[mask]) if np.any(mask) else 0

        # Append to result lists
        retention_times.append(rt)
        xic_intensities.append(xic_intensity)

    return np.array(retention_times), np.array(xic_intensities)

def plot_xic(retention_times, xic_intensities):
    """
    Plots the Extracted Ion Chromatogram (XIC) using Plotly.
    
    Parameters:
    retention_times (array): Array of retention times.
    xic_intensities (array): Array of intensities within the specified m/z range.
    """
    # Create a DataFrame for plotting
    xic_data = {"Retention Time": retention_times, "XIC Intensity": xic_intensities}
    
    # Plot XIC
    fig = px.line(xic_data, x="Retention Time", y="XIC Intensity", title="Extracted Ion Chromatogram (XIC)")
    fig.update_xaxes(title="Retention Time (min)")
    fig.update_yaxes(title="Intensity")
    fig.show()

# Usage
# Define the m/z range of interest, e.g., (500, 505)
mz_range = (500, 505)
rt_xic, intensity_xic = generate_xic(result_dict, mz_range)
plot_xic(rt_xic, intensity_xic)


Averaging a selection range

In [106]:
def average_intensity_in_range(ms_dict, rt_min, rt_max, ms_level="MS1"):
    """
    Averages intensity values within a specified retention time range for a given MS level.
    
    Parameters:
    ms_dict (dict): The dictionary generated from the mzML file.
    rt_min (float): The minimum retention time of the desired range.
    rt_max (float): The maximum retention time of the desired range.
    ms_level (str): The MS level to process (default is "MS1").

    Returns:
    tuple: Two arrays, one for the m/z values and one for the averaged intensities.
    """
    # Extract the MS level dictionary and filter retention times in range
    ms_level_dict = ms_dict.get(ms_level, {})
    rt_filtered = {
        rt: data for rt, data in ms_level_dict.items() if rt_min <= rt <= rt_max
    }

    # Collect all intensity arrays and m/z values
    all_mz_values = []
    all_intensity_values = []

    for data in rt_filtered.values():
        all_mz_values.append(data["m/z"])
        all_intensity_values.append(data["intensity"])

    # Align m/z values and calculate the average intensities
    if all_mz_values:
        # Find the common m/z range (use the first spectrum as reference)
        common_mz = all_mz_values[0]

        # Interpolate all intensities to match the reference m/z values
        interpolated_intensities = [
            np.interp(common_mz, mz_array, intensity_array)
            for mz_array, intensity_array in zip(all_mz_values, all_intensity_values)
        ]

        # Calculate the mean intensity at each m/z position
        average_intensity = np.mean(interpolated_intensities, axis=0)

        return common_mz, average_intensity
    else:
        return np.array([]), np.array([])

# Usage
# Define your RT range
rt_min, rt_max = 5.00, 6.00
mz_values, avg_intensity = average_intensity_in_range(result_dict, rt_min, rt_max)

In [ ]:
fig_avg_range = px.line(x=mz_values, y=avg_intensity)
fig_avg_range.show()

In [4]:
#finds the MS1 spectra for a given value or range.

# def get_ms_spectrum(result_dict, rt_value_or_range):
def get_ms_spectrum(result_dict, rt_value_or_range, mz_grid_resolution=0.01, ms_level="MS1"):
    """
    Finds and processes MS1 spectra for a given retention time range or a single retention time.
    
    Parameters:
    result_dict (dict): The dictionary with MS1 and MS2 data, containing retention time, m/z, and intensity values.
    rt_value_or_range (float or tuple): A single retention time (float) or a range (tuple of start and end retention times).
    
    Returns:
    dict: A dictionary with 'm/z' and 'intensity' arrays. If given a range, intensities are averaged.
    """
    ms_data = result_dict.get(ms_level, {})
    
    # if isinstance(rt_value_or_range, tuple):
    #     # Extract the retention time range
    #     rt_start, rt_end = rt_value_or_range

    #     # Filter retention times within the range and accumulate intensities and m/z values
    #     mz_values = []
    #     intensity_values = []

    #     for rt, data in ms_data.items():
    #         if rt_start <= rt <= rt_end:
    #             mz_values.append(data["m/z"])
    #             intensity_values.append(data["intensity"])

    #     # If we have multiple spectra within the range, average intensities
    #     if intensity_values:
    #         # Stack the m/z and intensity values to handle averaging
    #         all_mz = np.vstack(mz_values)
    #         all_intensity = np.vstack(intensity_values)
            
    #         # Average the intensities for each m/z
    #         avg_intensity = np.mean(all_intensity, axis=0)
    #         mz = all_mz[0]  # m/z values should be consistent

    #         return {"m/z": mz, "intensity": avg_intensity}
    #     else:
    #         raise ValueError("No data found within the specified retention time range.")
    if isinstance(rt_value_or_range, tuple):
        # Extract the retention time range
        rt_start, rt_end = rt_value_or_range

        # Filter retention times within the range and accumulate intensities and m/z values
        mz_values = []
        intensity_values = []

        for rt, data in ms_data.items():
            if rt_start <= rt <= rt_end:
                mz_values.append(data["m/z"])
                intensity_values.append(data["intensity"])

        # If we have multiple spectra within the range, interpolate and average intensities
        if intensity_values:
            # Create a common m/z grid over the range of observed m/z values with specified resolution
            min_mz = min(mz.min() for mz in mz_values)
            max_mz = max(mz.max() for mz in mz_values)
            common_mz_grid = np.arange(min_mz, max_mz, mz_grid_resolution)

            # Interpolate each intensity array onto the common m/z grid
            interpolated_intensities = []
            for mz, intensity in zip(mz_values, intensity_values):
                interpolated_intensity = np.interp(common_mz_grid, mz, intensity, left=0, right=0)
                interpolated_intensities.append(interpolated_intensity)
            
            # Average the interpolated intensities
            avg_intensity = np.mean(interpolated_intensities, axis=0)

            return {"m/z": common_mz_grid, "intensity": avg_intensity}
        else:
            raise ValueError("No data found within the specified retention time range.")
    

    elif isinstance(rt_value_or_range, (float, int)):
        # Single retention time provided
        rt = rt_value_or_range
        ms_idx = (result_dict[ms_level][rt] if rt in result_dict[ms_level] else result_dict[ms_level][min(result_dict[ms_level].keys(), key=lambda x: abs(x-rt))])
        return ms_idx
    
    else:
        raise TypeError("Input should be a single float/int retention time or a tuple (start, end) for a range.")

# Usage
# For a retention time range:
ms1_spectrum = get_ms_spectrum(result_dict, (11.9, 12.1),ms_level="MS1")
ms2_spectrum = get_ms_spectrum(result_dict, (11.9, 12.1),ms_level="MS2")
# For a single retention time:
# spectrum = get_ms_spectrum(result_dict, 5.0)


In [ ]:
fig_avg_range = px.line(x=ms1_spectrum['m/z'], y=ms1_spectrum['intensity'])
fig_avg_range.show()

In [ ]:
fig_avg_range = px.line(x=ms2_spectrum['m/z'], y=ms2_spectrum['intensity'])
fig_avg_range.show()

Adding figure labels section

In [ ]:
def find_top_peaks(mz_values, avg_intensity, min_distance=1.0, top_n=5):
    """
    Finds the top peaks in the intensity array, ensuring they are a minimum distance apart.
    
    Parameters:
    mz_values (array): Array of m/z values.
    avg_intensity (array): Array of intensity values.
    min_distance (float): Minimum distance in m/z between peaks.
    top_n (int): Number of top peaks to find.
    
    Returns:
    pd.DataFrame: DataFrame containing the m/z and intensity of the top peaks.
    """
    # Create a DataFrame for easier handling
    data = pd.DataFrame({"m/z": mz_values, "intensity": avg_intensity})

    # Sort by intensity in descending order
    data = data.sort_values(by="intensity", ascending=False).reset_index(drop=True)
    
    # List to store selected peaks
    selected_peaks = []
    
    for _, row in data.iterrows():
        mz = row["m/z"]
        intensity = row["intensity"]
        
        # Check if this peak is at least `min_distance` away from all selected peaks
        if all(abs(mz - peak["m/z"]) >= min_distance for peak in selected_peaks):
            selected_peaks.append({"m/z": mz, "intensity": intensity})
            
        # Stop if we have found the desired number of peaks
        if len(selected_peaks) >= top_n:
            break
    
    return pd.DataFrame(selected_peaks)

def plot_top_5_peaks(mz_values, avg_intensity):
    """
    Plots the m/z vs. averaged intensity values and labels the top 5 peaks.
    
    Parameters:
    mz_values (array): Array of m/z values.
    avg_intensity (array): Array of averaged intensity values.
    """
    # Find the top 5 peaks at least 1 m/z apart
    top_peaks = find_top_peaks(mz_values, avg_intensity, min_distance=1.0, top_n=10)

    # Create a DataFrame for the main line plot
    data = pd.DataFrame({"m/z": mz_values, "intensity": avg_intensity})

    # Create the plot
    fig = px.line(data, x="m/z", y="intensity")

    # Add annotations for each of the top 5 peaks
    for _, row in top_peaks.iterrows():
        fig.add_annotation(
            x=row["m/z"],
            y=row["intensity"],
            text=f"{row['m/z']:.4f}",
            showarrow=True,
            arrowhead=2,
            ax=0,
            ay=-40
        )

    # Show the plot
    fig.show()

# Usage
plot_top_5_peaks(mz_values, avg_intensity)

Intact mass Deconvolution using Unidec 

In [41]:
import numpy as np
import unidec as ud


